# 03. 시계열 분석 — 계정 표준화와 분기별 독립 데이터

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/dartlab/blob/master/notebooks/tutorials/03_timeseries.ipynb)

DART 원본 데이터는 회사마다 계정명이 다르고 분기 데이터는 누적값이다. 이 튜토리얼에서는 이 두 문제를 해결하는 DartLab의 시계열 엔진을 다룬다.

- 와 계정 표준화가 필요한가
- 시계열 생성 (`timeseries` property)
- 반환 구조 이해 (dict + period list)
- 누적 → 독립 분기 변환 원리
- TTM (Trailing Twelve Months)
- 연간 시계열
- 여러 기업 시계열 비교

📖 [문서 보기](https://eddmpython.github.io/dartlab/docs/tutorials/timeseries)

In [ ]:
!pip install -q dartlab

In [ ]:
from dartlab import Company

## 와 계정 표준화가 필요한가

DART에서 "매출"을 가져오면 회사마다 이름이 다르다.

```
삼성전자:  ifrs-full_Revenue      → 수익(매출액)
SK하이닉스: dart_OperatingRevenue  → 영업수익
현대차:    ifrs-full_Revenue      → 매출액
카카오:    dart_Revenue           → 수익(매출액)
```

DartLab은 **7단계 매핑 파이프라인**으로 이 문제를 해결한다. 결과적으로 모든 기업의 매출은 `revenue`, 영업이익은 `operating_income`이라는 하나의 키로 접근할 수 있다.

## 시계열 생성

In [ ]:
c = Company("005930")
series, periods = c.timeseries

print(f"기간 수: {len(periods)}")
print(f"첫 기간: {periods[0]}")
print(f"마지막 기간: {periods[-1]}")

## 반환 구조 이해

`series`는 재무제표별 dict의 dict이다. `periods`는 분기 레이블 리스트다.

```python
series = {
    "BS": { "total_assets": [...], "current_assets": [...], ... },
    "IS": { "revenue": [...], "operating_income": [...], ... },
    "CF": { "operating_cashflow": [...], ... }
}
```

In [ ]:
print("BS 계정:", list(series["BS"].keys())[:10])

In [ ]:
print("IS 계정:", list(series["IS"].keys())[:10])

In [ ]:
print("CF 계정:", list(series["CF"].keys())[:10])

In [ ]:
for i in range(-4, 0):
    period = periods[i]
    rev = series["IS"]["revenue"][i]
    if rev:
        print(f"{period}: {rev/1e8:,.0f}억원")

## 누적 → 독립 분기 변환

DART의 분기 보고서는 **누적 구조**로 제출된다.

```
누적                    독립 분기
Q1:         400    →    Q1:  400  (그대로)
Q1+Q2:    1,000    →    Q2:  600  (1,000 - 400)
Q1+Q2+Q3: 2,200    →    Q3: 1,200 (2,200 - 1,000)
연간:     3,000    →    Q4:  800  (3,000 - 2,200)
```

이 변환은 IS와 CF에만 적용된다. BS는 시점 잔액이므로 변환하지 않는다.

In [ ]:
print("이미 독립 분기 값이다:")
for i in range(-8, 0):
    period = periods[i]
    rev = series["IS"]["revenue"][i]
    if rev:
        print(f"  {period}: 매출 {rev/1e8:,.0f}억")

## 특정 계정 조회 함수

### getLatest — 가장 최근 값
### getTTM — 최근 4분기 합산 (Trailing Twelve Months)

In [ ]:
from dartlab.engines.dart.finance import getTTM, getLatest

assets = getLatest(series, "BS", "total_assets")
print(f"자산총계: {assets/1e8:,.0f}억원")

cash = getLatest(series, "BS", "cash_and_equivalents")
if cash:
    print(f"현금: {cash/1e8:,.0f}억원")

In [ ]:
rev_ttm = getTTM(series, "IS", "revenue")
oi_ttm = getTTM(series, "IS", "operating_income")

print(f"TTM 매출: {rev_ttm/1e8:,.0f}억원")
print(f"TTM 영업이익: {oi_ttm/1e8:,.0f}억원")

if rev_ttm and oi_ttm:
    margin = oi_ttm / rev_ttm * 100
    print(f"영업이익률: {margin:.1f}%")

## 연간 시계열

In [ ]:
from dartlab.engines.dart.finance import buildAnnual

annual_series, years = buildAnnual("005930")

for i, year in enumerate(years[-5:], len(years)-5):
    rev = annual_series["IS"]["revenue"][i]
    if rev:
        print(f"{year}: 매출 {rev/1e8:,.0f}억원")

## 여러 기업 시계열 비교

In [ ]:
companies = [
    Company("005930"),
    Company("000660"),
    Company("035420"),
]

for comp in companies:
    s, p = comp.timeseries
    rev = getTTM(s, "IS", "revenue")
    oi = getTTM(s, "IS", "operating_income")
    assets = getLatest(s, "BS", "total_assets")

    rev_str = f"{rev/1e8:,.0f}억" if rev else "N/A"
    oi_str = f"{oi/1e8:,.0f}억" if oi else "N/A"
    asset_str = f"{assets/1e8:,.0f}억" if assets else "N/A"

    print(f"{comp.corpName}: 매출 {rev_str} / 영업이익 {oi_str} / 자산 {asset_str}")

In [ ]:
samsung = Company("005930")
sk = Company("000660")

s1, p1 = samsung.timeseries
s2, p2 = sk.timeseries

for i in range(-8, 0):
    period = p1[i]
    r1 = s1["IS"]["revenue"][i]
    r2 = s2["IS"]["revenue"][i] if abs(i) <= len(s2["IS"]["revenue"]) else None

    r1_str = f"{r1/1e8:,.0f}억" if r1 else "N/A"
    r2_str = f"{r2/1e8:,.0f}억" if r2 else "N/A"
    print(f"{period}: 삼성전자 {r1_str} / SK하이닉스 {r2_str}")

---

다음: [4. 재무비율](./04_ratios.ipynb) — 시계열에서 자동 계산되는 재무비율